In [71]:
import pandas as pd
import json
from pathlib import Path
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os

In [72]:
#Load Dyanmic Paths
BASE_DIR = Path.cwd()
EMBED_DIR = BASE_DIR / "Embeddings"
IR_DIR = BASE_DIR / "IR2025"

DOCS_CSV = IR_DIR / "documents.csv"
QUERIES_CSV = IR_DIR / "queries.csv"
DOC_EMBED_FILE = EMBED_DIR / "ir2025_embeddings.npy"
QUERY_EMBED_FILE = EMBED_DIR / "query_embeddings.npy"
DOC_INDEXED_CSV = IR_DIR / "documents_with_index.csv"
QUERY_INDEXED_CSV = IR_DIR / "queries_with_index.csv"

EMBED_DIR.mkdir(exist_ok=True)
IR_DIR.mkdir(exist_ok=True)

In [73]:
def preprocess(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)                      # collapse spaces/newlines
    s = re.sub(r"http\S+|www\.\S+", "<URL>", s)     # replace URLs
    s = re.sub(r"\S+@\S+", "<EMAIL>", s)            # replace emails
    s = re.sub(r"<[^>]+>", " ", s)                  # remove HTML tags
    s = ''.join(ch for ch in s if ord(ch) >= 32)    # remove control chars
    s = re.sub(r"['\"]", "", s)                     # remove quotes
    return s.strip()


In [74]:
def build_or_load_embeddings(
    df,
    model,
    text_column="Text",
    embedding_file="Embeddings/ir2025_embeddings.npy",
    indexed_csv="IR2025/documents_with_index.csv",
    batch_size=64
):
    if os.path.exists(embedding_file) and os.path.exists(indexed_csv):
        print(f"🔁 Found existing embeddings → loading from '{embedding_file}'")
        embeddings = np.load(embedding_file)
        df = pd.read_csv(indexed_csv)
    else:
        print(f"⚙️ Generating embeddings using model: {model.__class__.__name__}")
        texts = df[text_column].astype(str).tolist()
        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        print("✅ Embeddings created successfully!")
        print("Shape:", embeddings.shape)

        np.save(embedding_file, embeddings)
        df["embedding_index"] = range(len(df))
        df.to_csv(indexed_csv, index=False)

        print(f"💾 Saved: {embedding_file}")
        print(f"💾 Saved: {indexed_csv}")

    return df, embeddings

In [75]:
df_docs = pd.read_csv(DOCS_CSV)
df_queries = pd.read_csv(QUERIES_CSV)

df_docs = df_docs.dropna(subset=["Text"])
df_docs["Text"] = df_docs["Text"].astype(str).map(preprocess)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
df_docs, doc_embeddings = build_or_load_embeddings(
    df_docs, model,
    embedding_file=str(DOC_EMBED_FILE),
    indexed_csv=str(DOC_INDEXED_CSV)
)


🔁 Found existing embeddings → loading from 'c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\Embeddings\ir2025_embeddings.npy'
🔁 Found existing embeddings → loading from 'c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\Embeddings\query_embeddings.npy'


In [ ]:
def build_faiss_index(doc_embeddings, use_cosine=True):
    doc_embeddings = doc_embeddings.astype("float32")
    if use_cosine:
        faiss.normalize_L2(doc_embeddings)
        index = faiss.IndexFlatIP(doc_embeddings.shape[1])
    else:
        index = faiss.IndexFlatL2(doc_embeddings.shape[1])
    index.add(doc_embeddings)
    print(f"FAISS index ready — {index.ntotal} documents, dim={doc_embeddings.shape[1]}, metric={'cosine' if use_cosine else 'L2'}")
    return index

index= build_faiss_index(doc_embeddings, use_cosine=True)

In [ ]:
df_queries, query_embeddings = build_or_load_embeddings(
    df_queries, model,
    embedding_file=str(QUERY_EMBED_FILE),
    indexed_csv=str(QUERY_INDEXED_CSV)
)

FAISS index ready — 18316 documents, dim=384, metric=cosine
